In [ ]:
import os
import numpy as np
import tensorflow as tf

from silx.io.dictdump import h5todict

In [ ]:
cpu = tf.config.list_physical_devices("CPU")[0]
tf.config.set_visible_devices(cpu)

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

path_to_figs = "./postproc/testing/"
os.makedirs(path_to_figs, exist_ok=True)


In [ ]:
path_to_model = "./saving/me_int/SavedModel/"
path_to_data = "./data/test.hdf5"
path_to_norm = "./data/norm/"
path_to_edges = "./data/fwd_edge_index.txt"

In [ ]:
# Model
model = tf.keras.models.load_model(path_to_model)
# Model - Node energy factor
model_enefac = tf.keras.Model(
    inputs=model.inputs,
    outputs=tf.gather(model.layers[4].output[0], indices=[2], axis=-1)
)
# Model - Node energy
model_energy = tf.keras.Model(
    inputs=model.inputs,
    outputs=tf.gather(model.layers[5].output[0], indices=[2], axis=-1)
)

In [ ]:
def get_data(data):
    # Collect data
    x = [data[i][1:] for i in ("node_attr", "edge_attr", "cond_attr")]
    y = data["target"][1:].squeeze()
    t = x[-1][:,0,0]
    # Make prediction
    x = tf.nest.map_structure(tf.convert_to_tensor, x)
    y_pred = model(x).numpy().squeeze()
    # > Node energy
    e_pred = model_energy(x).numpy().squeeze()
    e_fact = model_enefac(x).numpy().squeeze()
    e = e_pred / e_fact
    return t, y, y_pred, e, e_pred

In [ ]:
def plot_data(
    x=None,
    y=None,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$X$'],
    scales=['log', 'log'],
    names=None,
    figname=None
):
    # Initialize figures
    fig = plt.figure()
    ax = fig.add_subplot()
    # x axis
    xlim = [np.amin(x), np.amax(x)]
    ax.set_xlim(xlim)
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    true_style = dict(
        lw=1
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=y.shape[0]//20,
        ls='--',
        lw=1
    )
    for d in range(y.shape[1]):
        ax.plot(x, y[:,d], c=COLORS[d], **pred_style)
        ax.plot(xref, yref[:,d], c=COLORS[d], label=names[d], **true_style)
    plt.legend()
    if (figname is not None):
        fig.savefig(figname)
    plt.close()

In [ ]:
data = h5todict(path_to_data)
nodes_names = [r'O$_2^{(%d)}$' % (i+1) for i in range(3)]

In [ ]:
for (i, datai) in data.items():
    t, y, y_pred, e, e_pred = get_data(datai)
    kwargs=dict(
        x=t,
        y=y_pred,
        xref=t,
        yref=y,
        labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
        names=nodes_names
    )
    plot_data(
        scales=['log', 'log'],
        figname=path_to_figs + f"/{i}_log",
        **kwargs
    )
    plot_data(
        scales=['log', 'linear'],
        figname=path_to_figs + f"/{i}_lin",
        **kwargs
    )
    kwargs=dict(
        x=t,
        y=e_pred,
        xref=t,
        yref=e,
        labels=[r'$t$ [s]', r'$e$ [J]'],
        names=nodes_names
    )
    plot_data(
        scales=['log', 'log'],
        figname=path_to_figs + f"/{i}_energy",
        **kwargs
    )